# WEAC Skier Stability Criterion — All Pathways

This notebook demonstrates the end-to-end pipeline from snow pit observations to the
**WEAC coupled skier anticrack nucleation criterion** (Weißgraeber & Rosendahl 2023).

## What is g_delta?

> **g_delta** (Γ/Γc) is the dimensionless stability index from the coupled anticrack
> nucleation model.  It is the ratio of the mixed-mode energy release rate (ERR) to
> the critical mixed-mode ERR.  A value ≥ 1 means the anticrack can nucleate,
> i.e. the slab is potentially unstable.

## Requirements

```bash
pip install "snowpyt-mechparams[weac]"
```

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Graph Inspection](#2-graph-inspection)
3. [Build a Synthetic Slab](#3-build-a-synthetic-slab)
4. [Run `calculate_weac_skier` Directly](#4-run-calculate_weac_skier-directly)
5. [Inspect `WeacSkierResult`](#5-inspect-weacskierresult)
6. [Find All Pathways to g_delta](#6-find-all-pathways-to-g_delta)
7. [Run the Execution Engine — All Pathways at Once](#7-run-the-execution-engine--all-pathways-at-once)
8. [Real Dataset: ECTP Slabs](#8-real-dataset-ectp-slabs)
9. [g_delta Distribution by Pathway](#9-g_delta-distribution-by-pathway)

## 1. Setup & Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
from uncertainties import ufloat

# SnowPyt-MechParams
from snowpyt_mechparams.data_structures import Layer, Slab
from snowpyt_mechparams.data_structures.weak_layer import WeakLayer
from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig

# WEAC adapter
from snowpyt_mechparams.stability_models.weac import (
    calculate_weac_skier
)

print('Imports OK')

Imports OK


## 2. Graph Inspection

The dependency graph connects snow pit observations → layer mechanics → stability.

In [2]:
from snowpyt_mechparams.graph.definitions import (
    WEAK_LAYER_PARAMS, STABILITY_PARAMS
)

g_delta_node = graph.get_node('g_delta')
print(f'g_delta node:  {g_delta_node}')
print(f'Level:         {g_delta_node.level}')
print(f'Incoming edges: {[(e.start.parameter, e.method_name) for e in g_delta_node.incoming_edges]}')
print()
print(f'WEAK_LAYER_PARAMS: {sorted(WEAK_LAYER_PARAMS)}')
print(f'STABILITY_PARAMS:  {sorted(STABILITY_PARAMS)}')

g_delta node:  Node(type='parameter', parameter='g_delta', level='stability_model')
Level:         stability_model
Incoming edges: [('merge_weac_inputs', 'weac_skier')]

WEAK_LAYER_PARAMS: ['G_IIc', 'G_Ic', 'G_c', 'sigma_c', 'sigma_comp', 'tau_c']
STABILITY_PARAMS:  ['g_delta']


In [3]:
merge_node = graph.get_node('merge_weac_inputs')
print('merge_weac_inputs incoming edges (prerequisites):')
for edge in sorted(merge_node.incoming_edges, key=lambda e: e.start.parameter):
    print(f'  {edge.start.parameter:25s}  (level={edge.start.level})')

merge_weac_inputs incoming edges (prerequisites):
  G_IIc                      (level=weak_layer)
  G_Ic                       (level=weak_layer)
  G_c                        (level=weak_layer)
  density                    (level=layer)
  elastic_modulus            (level=layer)
  poissons_ratio             (level=layer)
  shear_modulus              (level=layer)
  sigma_c                    (level=weak_layer)
  sigma_comp                 (level=weak_layer)
  tau_c                      (level=weak_layer)


## 3. Build a Synthetic Slab

We construct a simple two-layer slab with hand-hardness and grain form data so the
execution engine can compute density, elastic modulus, Poisson's ratio, and shear modulus.

In [4]:
# Two-layer slab: rounded grains, typical alpine facet conditions
layers = [
    Layer(
        thickness=ufloat(15.0, 1.0),    # cm, top layer
        hand_hardness='1F',
        grain_form='RG',
        grain_size_avg=ufloat(0.5, 0.1),
    ),
    Layer(
        thickness=ufloat(20.0, 1.0),    # cm, bottom layer
        hand_hardness='4F',
        grain_form='FC',
        grain_size_avg=ufloat(1.5, 0.3),
    ),
]

# Weak layer below the slab (faceted crystals, common avalanche trigger)
weak_layer = Layer(
    thickness=ufloat(5.0, 0.5),     # cm
    density_measured=ufloat(120.0, 10.0),  # kg/m³
)

slab = Slab(
    layers=layers,
    angle=ufloat(38.0, 1.0),   # degrees
    weak_layer=weak_layer,
)

print(f'Slab: {len(slab.layers)} layers, angle = {slab.angle} °')
print(f'Total slab thickness: {slab.total_thickness} cm')

Slab: 2 layers, angle = 38.0+/-1.0 °
Total slab thickness: 35.0+/-1.4 cm


## 4. Run `calculate_weac_skier` Directly

The adapter converts a fully-populated `Slab` (layer mechanics + weak-layer params)
into WEAC inputs and returns a `WeacSkierResult`.

Before calling, we need to compute the mechanical parameters.  We'll use the
execution engine for that (see Section 7 for the full pipeline), but here we
manually assign typical values for a quick demonstration.

In [5]:
# Pre-populate mechanical parameters (layer-level)
for layer in slab.layers:
    layer.density_calculated = ufloat(250.0, 20.0)   # kg/m³
    layer.elastic_modulus    = ufloat(5.0, 0.5)      # MPa
    layer.poissons_ratio     = ufloat(0.2, 0.02)     # dimensionless
    layer.shear_modulus      = ufloat(2.0, 0.2)      # MPa

# Fracture / strength parameters (Weißgraeber & Rosendahl 2023 reference values)
slab.weac_layer = WeakLayer(
    G_c=ufloat(1.0, 0.0),
    G_Ic=ufloat(0.56, 0.0),
    G_IIc=ufloat(0.79, 0.0),
    sigma_c=ufloat(6.16, 0.0),
    tau_c=ufloat(5.09, 0.0),
    sigma_comp=ufloat(2.6, 0.0),
)

print('Slab ready for WEAC')

Slab ready for WEAC


In [6]:
result = calculate_weac_skier(slab, skier_mass=80.0)

if result is None:
    print('ERROR: Missing required inputs — check slab fields.')
else:
    stability = 'UNSTABLE' if result.g_delta >= 1.0 else 'STABLE'
    print(f'g_delta = {result.g_delta:.4f}  →  {stability}')
    print(f'Converged: {result.converged}')

g_delta = 23.8179  →  UNSTABLE
Converged: True


## 5. Inspect `WeacSkierResult`

The result object contains all outputs from the WEAC coupled criterion.

In [7]:
if result is not None:
    fields = [
        ('g_delta',               'dimensionless',  'Primary stability index (≥1 → unstable)'),
        ('converged',             'bool',           'Whether the iterative solver converged'),
        ('G_I',                   'J/m²',           'Mode-I ERR at critical anticrack size'),
        ('G_II',                  'J/m²',           'Mode-II ERR at critical anticrack size'),
        ('G_total',               'J/m²',           'Total mixed-mode ERR'),
        ('critical_skier_weight', 'N',              'Load at onset of anticrack nucleation'),
        ('crack_length',          'mm',             'Critical anticrack half-length'),
        ('max_dist_stress',       'kPa',            'Max distributed stress in weak layer'),
        ('min_dist_stress',       'kPa',            'Min distributed stress in weak layer'),
        ('dist_ERR_envelope',     'J/m²',           'Distributed load ERR envelope'),
        ('segment_length',        'mm',             'Half-segment length used'),
        ('skier_mass',            'kg',             'Skier mass used'),
    ]
    print(f'  {"Field":<28s}  {"Value":>12s}  {"Unit":<14s}  Description')
    print(f'  {"-"*90}')
    for field, unit, desc in fields:
        val = getattr(result, field)
        if isinstance(val, float):
            val_str = f'{val:.4f}'
        else:
            val_str = str(val)
        print(f'  {field:<28s}  {val_str:>12s}  {unit:<14s}  {desc}')

  Field                                Value  Unit            Description
  ------------------------------------------------------------------------------------------
  g_delta                            23.8179  dimensionless   Primary stability index (≥1 → unstable)
  converged                             True  bool            Whether the iterative solver converged
  G_I                                 1.0558  J/m²            Mode-I ERR at critical anticrack size
  G_II                               -0.0060  J/m²            Mode-II ERR at critical anticrack size
  G_total                             1.0498  J/m²            Total mixed-mode ERR
  critical_skier_weight             241.1679  N               Load at onset of anticrack nucleation
  crack_length                        1.0000  mm              Critical anticrack half-length
  max_dist_stress                     1.0090  kPa             Max distributed stress in weak layer
  min_dist_stress                     0.0217  kPa     

## 6. Find All Pathways to g_delta

The parameterization algorithm enumerates all valid computation pathways from
snow pit data to `g_delta`.

Each pathway is a unique combination of:
- **Density method** (4 options)
- **Elastic modulus method** (4 options)
- **Poisson's ratio method** (2 options)
- **Shear modulus method** (1 option)
- **Weak-layer fracture params method** (1 option — weissgraeber_rosendahl)
- **Stability criterion** (1 option — weac_skier)

Total: 4 × 4 × 2 × 1 × 1 × 1 = **32 pathways**

In [8]:
g_delta_node = graph.get_node('g_delta')
pathways = find_parameterizations(graph, g_delta_node)

print(f'Found {len(pathways)} pathways for g_delta:\n')
for i, pathway in enumerate(pathways[:5], 1):   # show first 5
    print(f'Pathway {i}:')
    print(pathway)
    print()
if len(pathways) > 5:
    print(f'  ... and {len(pathways) - 5} more')

Found 32 pathways for g_delta:

Pathway 1:
branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_weac_inputs
branch 2: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_density_grain_form
branch 3: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_density_grain_form
branch 4: snow_pit -- data_flow --> measured_grain_form -- kochle --> poissons_ratio -- data_flow --> merge_weac_inputs
branch 5: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_density_grain_form
branch 6: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_density_grain_form
branch 7: snow_pit -- weissgraeber_rosendahl --> G_c -- data_flow --> merge_weac_inputs
branch 8: snow_pit -- weissgraeber_rosendahl --> G_Ic -- data_flow --> merge_weac_inputs
branch 9: snow_pit -- weissgraeber_rosendahl --> G_IIc -- data_flow --> merge_weac_inputs
branch 10: snow_pit -

## 7. Run the Execution Engine — All Pathways at Once

The `ExecutionEngine` applies all pathways to a slab in one call, automatically
computing density → E/ν/G → weak-layer fracture params → g_delta.

**Note:** The slab below has only hand-hardness/grain-form data, not pre-computed
mechanical parameters.  The engine fills everything in.

In [9]:
# Fresh slab — only field observations, no computed parameters
slab_fresh = Slab(
    layers=[
        Layer(
            thickness=ufloat(15.0, 1.0),
            hand_hardness='1F',
            grain_form='RG',
            grain_size_avg=ufloat(0.5, 0.1),
        ),
        Layer(
            thickness=ufloat(20.0, 1.0),
            hand_hardness='4F',
            grain_form='FC',
            grain_size_avg=ufloat(1.5, 0.3),
        ),
    ],
    angle=ufloat(38.0, 1.0),
    weak_layer=Layer(
        thickness=ufloat(5.0, 0.5),
        density_measured=ufloat(120.0, 10.0),
    ),
)
print('Fresh slab created (no pre-computed mechanical params)')

Fresh slab created (no pre-computed mechanical params)


In [10]:
engine = ExecutionEngine(graph)
config = ExecutionConfig(include_method_uncertainty=True)

all_results = engine.execute_all(slab_fresh, 'g_delta', config=config)

successful = {pid: pr for pid, pr in all_results.pathways.items() if pr.success}
print(f'Total pathways tried:  {len(all_results.pathways)}')
print(f'Successful pathways:   {len(successful)}')
print(f'Failed pathways:       {len(all_results.pathways) - len(successful)}')

Total pathways tried:  32
Successful pathways:   8
Failed pathways:       24


In [11]:
rows = []
for pathway_result in successful.values():
    mu = pathway_result.methods_used
    weac_result = pathway_result.slab.weac_result
    if weac_result is None:
        continue
    rows.append({
        'density':          mu.get('density', '—'),
        'elastic_modulus':  mu.get('elastic_modulus', '—'),
        'poissons_ratio':   mu.get('poissons_ratio', '—'),
        'g_delta':          round(weac_result.g_delta, 4),
        'converged':        weac_result.converged,
        'G_total (J/m²)':   round(weac_result.G_total, 4),
        'crack_len (mm)':   round(weac_result.crack_length, 2),
    })

df = pd.DataFrame(rows).sort_values('g_delta')
print(df.to_string(index=False))

            density elastic_modulus poissons_ratio  g_delta  converged  G_total (J/m²)  crack_len (mm)
         geldsetzer       schottner         kochle   4.4904       True          0.7493             1.0
         geldsetzer       schottner     srivastava   4.5274       True          0.7507             1.0
kim_jamieson_table2       schottner         kochle   4.6165       True          0.7518             1.0
kim_jamieson_table2       schottner     srivastava   4.6538       True          0.7531             1.0
kim_jamieson_table2         wautier     srivastava  24.1526       True          1.0572             1.0
kim_jamieson_table2         wautier         kochle  24.1533       True          1.0572             1.0
         geldsetzer         wautier     srivastava  24.1650       True          1.0574             1.0
         geldsetzer         wautier         kochle  24.1657       True          1.0574             1.0


## 8. Real Dataset: ECTP Slabs

Load the bundled CAAML snow pit files and run all pathways over every ECTP slab.

In [12]:
from snowpyt_mechparams.snowpilot import parse_caaml_directory
from snowpyt_mechparams.data_structures import Pit

snow_pits_raw = parse_caaml_directory(str(Path('data')))
pits = [Pit.from_snow_pit(sp) for sp in snow_pits_raw]
print(f'Loaded {len(pits)} snow pits')

ectp_slabs = []
for pit in pits:
    for slab in pit.create_slabs(weak_layer_def='ECTP_failure_layer'):
        ectp_slabs.append(slab)
print(f'Created {len(ectp_slabs)} ECTP slabs')

Loaded 50278 snow pits
Created 14776 ECTP slabs


In [13]:
from tqdm import tqdm

engine   = ExecutionEngine(graph)
# weac_timeout_seconds: drop any slab whose WEAC solver takes longer than this.
# The coupled criterion occasionally recurses deeply on numerically difficult
# inputs; a 5-second budget keeps the batch moving without losing many slabs.
config   = ExecutionConfig(include_method_uncertainty=False, weac_timeout_seconds=5.0)

# Accumulate g_delta per pathway across all slabs
pathway_g_delta = {}   # {pathway_label: [g_delta_float, ...]}

for slab in tqdm(ectp_slabs, desc='Running g_delta', unit='slab'):
    results = engine.execute_all(slab, 'g_delta', config=config)
    for pr in results.pathways.values():
        if not pr.success or pr.slab.weac_result is None:
            continue
        mu    = pr.methods_used
        label = f"{mu.get('density','?')} → {mu.get('elastic_modulus','?')} → {mu.get('poissons_ratio','?')}"
        pathway_g_delta.setdefault(label, []).append(pr.slab.weac_result.g_delta)

# Summary table
summary = []
total   = len(ectp_slabs)
for label, vals in sorted(pathway_g_delta.items(), key=lambda x: -len(x[1])):
    arr = np.array(vals)
    summary.append({
        'Pathway':         label,
        'N slabs':         len(arr),
        'Coverage':        f'{len(arr)/total:.1%}',
        'Median g_delta':  round(float(np.median(arr)), 3),
        'Mean g_delta':    round(float(np.mean(arr)), 3),
        '% unstable':      f'{100*np.mean(arr >= 1.0):.1f}%',
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

Running g_delta: 100%|██████████| 14776/14776 [22:41<00:00, 10.85slab/s]  

                                     Pathway  N slabs Coverage  Median g_delta  Mean g_delta % unstable
              data_flow → schottner → kochle       12     0.1%           1.001         2.367      58.3%
          data_flow → schottner → srivastava       11     0.1%           1.000         2.623      45.5%
             geldsetzer → schottner → kochle        9     0.1%           1.001         2.575      66.7%
    kim_jamieson_table2 → schottner → kochle        9     0.1%           1.000         2.510      77.8%
           data_flow → bergfeld → srivastava        7     0.0%           1.596         4.858      85.7%
                data_flow → wautier → kochle        6     0.0%           1.001         1.001     100.0%
               geldsetzer → wautier → kochle        6     0.0%           1.000         1.000      50.0%
      kim_jamieson_table2 → wautier → kochle        6     0.0%           1.000         4.985      50.0%
              geldsetzer → bergfeld → kochle        5     0.0%  

## 9. g_delta Distribution by Pathway

Violin plots comparing the g_delta distributions across all pathways.
The red dashed line at g_delta = 1 marks the instability threshold.

In [14]:
import plotly.graph_objects as go

DENSITY_COLORS = {
    'geldsetzer':          'rgba( 68, 114, 196, 0.75)',
    'kim_jamieson_table2': 'rgba( 84, 168, 104, 0.75)',
    'kim_jamieson_table5': 'rgba(148, 103, 189, 0.75)',
    'data_flow':           'rgba(196, 140,  68, 0.75)',
}

# Sort pathways by coverage (most successful first)
sorted_labels = sorted(pathway_g_delta.keys(), key=lambda k: -len(pathway_g_delta[k]))

fig = go.Figure()

for label in reversed(sorted_labels):   # reversed so highest is at top
    vals = np.array(pathway_g_delta[label])
    n    = len(vals)
    density_method = label.split(' → ')[0]
    color      = DENSITY_COLORS.get(density_method, 'rgba(128,128,128,0.7)')
    line_color = color.replace('0.75', '1.0')
    pct        = n / total

    fig.add_trace(go.Violin(
        x=vals,
        y=[f"{label}  <i>({n:,} / {total:,}, {pct:.1%})</i>"] * n,
        orientation='h',
        name=label,
        box_visible=True,
        meanline_visible=True,
        points=False,
        fillcolor=color,
        line_color=line_color,
        opacity=0.85,
        showlegend=False,
    ))

# Instability threshold line
fig.add_vline(
    x=1.0,
    line_dash='dash',
    line_color='rgba(196,84,78,0.8)',
    line_width=2,
    annotation_text='g_delta = 1 (instability threshold)',
    annotation_position='top right',
)

fig.update_layout(
    title=dict(
        text=(
            '<b>g_delta Distribution by Calculation Pathway</b><br>'
            '<sup>ECTP slabs — ordered top→bottom by coverage — '
            'box = IQR, line = mean — colour = density method</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    xaxis=dict(
        title='g_delta  (Γ/Γ_c)',
        gridcolor='rgba(200,200,200,0.4)',
        zeroline=True,
        zerolinecolor='rgba(180,180,180,0.5)',
    ),
    yaxis=dict(
        autorange='reversed',
        tickfont=dict(size=9.5),
    ),
    plot_bgcolor='white',
    violingap=0.06,
    violingroupgap=0,
    width=1050,
    height=1380,
    margin=dict(l=420, r=40, t=100, b=60),
)

fig.show()